# 06 — Final submissions
Two submissions, both selected on OOF macro-F1 (never the public LB). Each file is named after the model/algorithm that produced it:
- **sub01 (primary)** — the best-OOF configuration. The blend search selected CatBoost alone and the threshold search returned no change, so this is the tuned **CatBoost** (gradient-boosted decision trees). OOF macro-F1 ≈ 0.829.
- **sub02 (diverse safety)** — an equal-weight **soft-voting ensemble** of all four learners (CatBoost + RandomForest + ExtraTrees + GradientBoosting). Slightly lower OOF, but a genuinely different prediction — a hedge for the private 70% split.

In [1]:
# --- Shared toolbox (identical across notebooks; see build_notebooks.py) ---
import warnings, json
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report, confusion_matrix

SEED = 42
N_FOLDS = 5
CLASS_NAMES = {0: "Wake", 1: "Light", 2: "Deep", 3: "REM"}
CLASSES = np.array([0, 1, 2, 3])
EOG = "eog_burst_index"            # the only column with missing values (~50%)

RAW_FEATURES = [
    "eeg_delta_power", "eeg_theta_power", "eeg_alpha_power", "eeg_sigma_power",
    "eeg_beta_power", "eeg_gamma_power", "eeg_slow_osc_power", "eeg_spectral_entropy",
    "eeg_spindle_density", "eeg_kcomplex_rate", "emg_chin_tone", "emg_tone_variance",
    "eog_movement_density", "eog_amplitude", "heart_rate_mean", "heart_rate_variability",
    "respiration_rate", "respiration_variability", "spo2_mean", "body_movement_index",
    EOG,
]
POWER = ["eeg_delta_power", "eeg_theta_power", "eeg_alpha_power", "eeg_sigma_power",
         "eeg_beta_power", "eeg_gamma_power", "eeg_slow_osc_power"]

HERE = Path.cwd()
ART = HERE / "artifacts"; ART.mkdir(exist_ok=True)
SUB = HERE / "submissions"; SUB.mkdir(exist_ok=True)


def load_data():
    """Return (train_df, test_df). Features kept as-is (NaN preserved)."""
    tr = pd.read_csv(HERE / "train.csv")
    te = pd.read_csv(HERE / "test.csv")
    return tr, te


def macro_f1(y_true, y_pred):
    """Competition metric: macro-averaged F1 over the 4 classes."""
    return f1_score(y_true, y_pred, average="macro")


def per_class_f1(y_true, y_pred):
    f = f1_score(y_true, y_pred, average=None, labels=CLASSES)
    return {CLASS_NAMES[c]: round(float(f[i]), 4) for i, c in enumerate(CLASSES)}


def softplus(x):
    """Numerically stable log(1+exp(x)); strictly positive and monotonic.
    Used to turn z-scored band powers (~50% negative) into positive magnitudes
    so band ratios are well-defined instead of dividing by near-zero."""
    x = np.asarray(x, dtype=float)
    return np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0.0)


def _aligned_proba(model, X):
    """predict_proba with columns aligned to CLASSES = [0,1,2,3]."""
    p = model.predict_proba(X)
    cls = list(np.asarray(model.classes_))
    idx = [cls.index(c) for c in CLASSES]
    return p[:, idx]


def run_oof(make_model, X, y, X_test, folds, needs_impute=False, use_eval_set=False):
    """Out-of-fold training under fixed folds.

    Returns (oof, test_p, oof_macro, fold_scores):
      oof     : (n_train, 4) out-of-fold probabilities (each row predicted once)
      test_p  : (n_test, 4) test probabilities, averaged over the 5 fold-models
      oof_macro: global macro-F1 over the assembled OOF matrix (primary metric)

    Two model families, identical folds:
      - CatBoost (needs_impute=False): NaN passed through natively.
      - sklearn trees (needs_impute=True): add EOG-missing flag + fill EOG NaN
        with the TRAIN-FOLD median (fit on train fold only -> no leakage)."""
    n = len(y)
    oof = np.zeros((n, len(CLASSES)))
    test_p = np.zeros((len(X_test), len(CLASSES)))
    fold_scores = []
    for tr_idx, va_idx in folds:
        Xtr, Xva, Xte = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy(), X_test.copy()
        ytr, yva = y[tr_idx], y[va_idx]
        if needs_impute:
            med = Xtr[EOG].median()
            for d in (Xtr, Xva, Xte):
                if EOG + "_missing" not in d.columns:
                    d[EOG + "_missing"] = d[EOG].isna().astype("int8")
                d[EOG] = d[EOG].fillna(med)
            assert not Xtr.isna().any().any(), "NaN remained after impute"
        model = make_model()
        if use_eval_set:
            model.fit(Xtr, ytr, eval_set=(Xva, yva))
        else:
            model.fit(Xtr, ytr)
        oof[va_idx] = _aligned_proba(model, Xva)
        test_p += _aligned_proba(model, Xte) / len(folds)
        fold_scores.append(macro_f1(yva, oof[va_idx].argmax(1)))
    oof_macro = macro_f1(y, oof.argmax(1))
    return oof, test_p, oof_macro, fold_scores


def load_folds():
    """Load the fixed StratifiedKFold split saved by 02_baseline."""
    d = np.load(ART / "folds.npz", allow_pickle=True)
    return [(d[f"tr{i}"], d[f"va{i}"]) for i in range(N_FOLDS)]


In [2]:
def log_result(step, model, feature_set, oof_macro, pcf, notes=""):
    """Write one row to results_log.csv. Idempotent per (step, model): re-running
    a notebook replaces its own row rather than duplicating it."""
    import csv
    path = HERE / "results_log.csv"
    header = ["step", "model", "feature_set", "oof_macro_f1",
              "f1_Wake", "f1_Light", "f1_Deep", "f1_REM", "notes"]
    rows = []
    if path.exists():
        with open(path, newline="") as fh:
            data = list(csv.reader(fh))
        if data and data[0] == header:
            rows = [r for r in data[1:] if not (len(r) >= 2 and r[0] == step and r[1] == model)]
    row = [step, model, feature_set, round(float(oof_macro), 5),
           pcf["Wake"], pcf["Light"], pcf["Deep"], pcf["REM"], notes]
    with open(path, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(header); w.writerows(rows); w.writerow(row)
    print("logged:", step, model, "OOF macro-F1 =", round(float(oof_macro), 5))


In [3]:
y = np.load(ART / "y_train.npy")
test_ids = np.load(ART / "test_ids.npy")
blend_oof = np.load(ART / "blend_oof.npy")
blend_test = np.load(ART / "blend_test.npy")
mult = np.array(json.load(open(ART / "thresholds.json"))["multipliers"])
blend_info = json.load(open(ART / "blend.json"))
MODELS = ["catboost_tuned", "rf", "et", "gb"]
oofs = {m: np.load(ART / f"{m}_oof.npy") for m in MODELS}
tests = {m: np.load(ART / f"{m}_test.npy") for m in MODELS}
print("blend weights:", blend_info["weights"], "| thresholds:", mult)

blend weights: {'catboost_tuned': 1.0, 'rf': 0.0, 'et': 0.0, 'gb': 0.0} | thresholds: [1. 1. 1. 1.]


## sub01 — best-OOF model (named for the algorithm actually selected)
The filename is built from what the pipeline actually chose, so it never lies: the blend members with non-zero weight, plus `_ThresholdTuned` only if the threshold search changed anything.

In [4]:
ALGO = {"catboost_tuned": "CatBoost", "rf": "RandomForest",
        "et": "ExtraTrees", "gb": "GradientBoosting"}
used = [m for m in blend_info["members"] if blend_info["weights"][m] > 0]
if len(used) == 1:
    tag = ALGO[used[0]] + ("_GBDT" if used[0] == "catboost_tuned" else "")
else:
    tag = "SoftVotingEnsemble-" + "-".join(ALGO[m] for m in used)
tag += "_ThresholdTuned" if not np.allclose(mult, 1.0) else ""
name1 = f"sub01_{tag}.csv"
pred1 = (blend_test * mult).argmax(1).astype(int)
sub1 = pd.DataFrame({"id": test_ids, "sleep_stage": pred1})
assert sub1.shape == (5000, 2)
assert sub1["id"].tolist() == list(range(9000, 14000))
assert sub1["sleep_stage"].isin(CLASSES).all()
sub1.to_csv(SUB / name1, index=False)
oof1 = macro_f1(y, (blend_oof * mult).argmax(1))
print(f"sub01 -> {name1} | OOF macro-F1:", round(oof1, 5))
print("class counts:", sub1["sleep_stage"].value_counts().sort_index().to_dict())
log_result("06_submit", tag, "+".join(used), oof1,
           per_class_f1(y, (blend_oof * mult).argmax(1)), "sub01 (primary): " + name1)

sub01 -> sub01_CatBoost_GBDT.csv | OOF macro-F1: 0.82898
class counts: {0: 1113, 1: 1310, 2: 1274, 3: 1303}
logged: 06_submit CatBoost_GBDT OOF macro-F1 = 0.82898


## sub02 — soft-voting ensemble (diverse safety net)
Equal-weight average of the four models' probabilities — a soft `VotingClassifier`. Genuinely different from CatBoost alone.

In [5]:
ens_oof = sum(oofs[m] for m in MODELS) / len(MODELS)
ens_test = sum(tests[m] for m in MODELS) / len(MODELS)
name2 = "sub02_SoftVotingEnsemble_CatBoost-RandomForest-ExtraTrees-GradientBoosting.csv"
pred2 = ens_test.argmax(1).astype(int)
sub2 = pd.DataFrame({"id": test_ids, "sleep_stage": pred2})
assert sub2.shape == (5000, 2) and sub2["sleep_stage"].isin(CLASSES).all()
sub2.to_csv(SUB / name2, index=False)
oof2 = macro_f1(y, ens_oof.argmax(1))
print(f"sub02 -> {name2} | OOF macro-F1:", round(oof2, 5))
print("class counts:", sub2["sleep_stage"].value_counts().sort_index().to_dict())
print("sub01 vs sub02 differ on",
      int((sub1["sleep_stage"] != sub2["sleep_stage"]).sum()), "of 5000 rows")
log_result("06_submit", "SoftVotingEnsemble", "+".join(MODELS), oof2,
           per_class_f1(y, ens_oof.argmax(1)), "sub02 (diverse safety): " + name2)

sub02 -> sub02_SoftVotingEnsemble_CatBoost-RandomForest-ExtraTrees-GradientBoosting.csv | OOF macro-F1: 0.81952
class counts: {0: 1098, 1: 1329, 2: 1274, 3: 1299}
sub01 vs sub02 differ on 206 of 5000 rows
logged: 06_submit SoftVotingEnsemble OOF macro-F1 = 0.81952


## Done
Upload limit is 20/day. **Primary = `sub01_CatBoost_GBDT.csv`** (best OOF ≈ 0.829); **safety = `sub02_SoftVotingEnsemble_*.csv`** (diverse hedge). Both chosen by OOF macro-F1, not the public LB. See `results_log.csv` for the full evidence table.